In [1]:
import os
import pandas as pd
import numpy as np

In [2]:

# ============================
# 0. PATH CONFIG
# ============================

BASE_DIR = "/Users/nishantkhandhar/Desktop/Fall 25/DPA/Project/CSP571---Final-Project"
DATASET_DIR = os.path.join(BASE_DIR, "Dataset")
CLEANED_DIR = os.path.join(BASE_DIR, "CleanedDataset")

os.makedirs(CLEANED_DIR, exist_ok=True)

crime_files = {
    2019: "Crimes_-_2019_20251111.csv",
    2020: "Crimes_-_2020_20251111.csv",
    2021: "Crimes_-_2021_20251111.csv",
    2022: "Crimes_-_2022_20251111.csv",
    2023: "Crimes_-_2023_20251111.csv",
}

HARDSHIP_FILE = "HardshipIndex.csv"

In [3]:
# ============================
# 1. LOAD + CLEAN CRIME DATA
# ============================

crime_frames = []

for year, filename in crime_files.items():
    path = os.path.join(DATASET_DIR, filename)
    print(f"Loading {year} from: {path}")

    df = pd.read_csv(path, low_memory=False)

    # ---- Standardize column names if they exist ----
    rename_map = {
        "Primary Type": "primary_type",
        "Primary_Type": "primary_type",
        "Community Area": "community_area",
        "COMMUNITY_AREA": "community_area",
        "Community Area Name": "community_area_name",
        "COMMUNITY_AREA_NAME": "community_area_name",
        "Date": "date",
        "DATE": "date",
        "Arrest": "arrest",
        "ARREST": "arrest",
        "Block": "block",
        "BLOCK": "block",
    }
    df = df.rename(columns=rename_map)

    # ---- Parse date ----
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df["year"] = df["date"].dt.year
        # Fallback: if year is NaN (e.g., parse failure), use the known table year
        df["year"] = df["year"].fillna(year).astype(int)
    else:
        # If there's no date column (unlikely), just assign year
        df["year"] = int(year)
        df["date"] = pd.NaT

    # ---- Keep only useful columns (drop everything else) ----
    keep_cols = [
        "year",
        "date",
        "primary_type",
        "arrest",
        "community_area",
        "community_area_name",
    ]
    # Some columns may be missing; keep intersection
    keep_cols = [c for c in keep_cols if c in df.columns]
    df = df[keep_cols]

    # ---- Drop rows with missing community_area ----
    if "community_area" not in df.columns:
        raise ValueError(f"'community_area' column missing in {filename}")

    df = df.dropna(subset=["community_area"])

    # ---- Ensure community_area is integer ----
    df["community_area"] = (
        df["community_area"]
        .astype(str)
        .str.strip()
        .str.replace(".0", "", regex=False)
    )
    df["community_area"] = df["community_area"].astype(int)

    # Optional: standardize primary_type to uppercase (helps grouping)
    if "primary_type" in df.columns:
        df["primary_type"] = df["primary_type"].astype(str).str.upper()

    # Save cleaned per-year file
    yearly_out = os.path.join(CLEANED_DIR, f"Crimes_Cleaned_{year}.csv")
    df.to_csv(yearly_out, index=False)
    print(f"Saved cleaned {year} crimes to: {yearly_out} (rows: {len(df)})")

    crime_frames.append(df)

Loading 2019 from: /Users/nishantkhandhar/Desktop/Fall 25/DPA/Project/CSP571---Final-Project/Dataset/Crimes_-_2019_20251111.csv


/var/folders/2t/r1cg59kd45qgtchr9ql9jmc40000gn/T/ipykernel_16930/2592619706.py:32: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["date"] = pd.to_datetime(df["date"], errors="coerce")


Saved cleaned 2019 crimes to: /Users/nishantkhandhar/Desktop/Fall 25/DPA/Project/CSP571---Final-Project/CleanedDataset/Crimes_Cleaned_2019.csv (rows: 261659)
Loading 2020 from: /Users/nishantkhandhar/Desktop/Fall 25/DPA/Project/CSP571---Final-Project/Dataset/Crimes_-_2020_20251111.csv


/var/folders/2t/r1cg59kd45qgtchr9ql9jmc40000gn/T/ipykernel_16930/2592619706.py:32: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["date"] = pd.to_datetime(df["date"], errors="coerce")


Saved cleaned 2020 crimes to: /Users/nishantkhandhar/Desktop/Fall 25/DPA/Project/CSP571---Final-Project/CleanedDataset/Crimes_Cleaned_2020.csv (rows: 212639)
Loading 2021 from: /Users/nishantkhandhar/Desktop/Fall 25/DPA/Project/CSP571---Final-Project/Dataset/Crimes_-_2021_20251111.csv


...

Saved cleaned 2021 crimes to: /Users/nishantkhandhar/Desktop/Fall 25/DPA/Project/CSP571---Final-Project/CleanedDataset/Crimes_Cleaned_2021.csv (rows: 209568)
Loading 2022 from: /Users/nishantkhandhar/Desktop/Fall 25/DPA/Project/CSP571---Final-Project/Dataset/Crimes_-_2022_20251111.csv


/var/folders/2t/r1cg59kd45qgtchr9ql9jmc40000gn/T/ipykernel_16930/2592619706.py:32: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["date"] = pd.to_datetime(df["date"], errors="coerce")


Saved cleaned 2022 crimes to: /Users/nishantkhandhar/Desktop/Fall 25/DPA/Project/CSP571---Final-Project/CleanedDataset/Crimes_Cleaned_2022.csv (rows: 239916)
Loading 2023 from: /Users/nishantkhandhar/Desktop/Fall 25/DPA/Project/CSP571---Final-Project/Dataset/Crimes_-_2023_20251111.csv


/var/folders/2t/r1cg59kd45qgtchr9ql9jmc40000gn/T/ipykernel_16930/2592619706.py:32: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["date"] = pd.to_datetime(df["date"], errors="coerce")


Saved cleaned 2023 crimes to: /Users/nishantkhandhar/Desktop/Fall 25/DPA/Project/CSP571---Final-Project/CleanedDataset/Crimes_Cleaned_2023.csv (rows: 263155)


In [4]:
# ============================
# 2. CONCATENATE ALL YEARS
# ============================

crime_all = pd.concat(crime_frames, ignore_index=True)
all_out = os.path.join(CLEANED_DIR, "Crimes_All_2019_2023_Cleaned.csv")
crime_all.to_csv(all_out, index=False)
print(f"Saved unified cleaned crime file to: {all_out} (rows: {len(crime_all)})")

Saved unified cleaned crime file to: /Users/nishantkhandhar/Desktop/Fall 25/DPA/Project/CSP571---Final-Project/CleanedDataset/Crimes_All_2019_2023_Cleaned.csv (rows: 1186937)


In [5]:
# ============================
# 3. LOAD + CLEAN HARDSHIP INDEX
# ============================

hardship_path = os.path.join(DATASET_DIR, HARDSHIP_FILE)
print(f"Loading Hardship Index from: {hardship_path}")

hardship = pd.read_csv(hardship_path)

# Rename columns
hardship = hardship.rename(columns={
    "Area Number": "community_area",
    "Community": "community_area_name",
    "Unemployment Rate": "unemployment_rate",
    "Hardship Index Value": "hardship_index",
    "Per Capita Income": "per_capita_income",
})

# STEP 1 — Remove footer / non-numeric rows
# Keep only rows where community_area is a number
hardship = hardship[hardship["community_area"].astype(str).str.isdigit()]

# STEP 2 — Convert to int now (won't break anymore)
hardship["community_area"] = hardship["community_area"].astype(int)

# STEP 3 — Clean unemployment rate
hardship["unemployment_rate"] = (
    hardship["unemployment_rate"]
    .astype(str)
    .str.replace("%", "", regex=False)
    .str.strip()
)
hardship["unemployment_rate"] = pd.to_numeric(
    hardship["unemployment_rate"], errors="coerce"
)

# STEP 4 — Clean Per Capita Income (remove $ and commas)
hardship["per_capita_income"] = (
    hardship["per_capita_income"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)
hardship["per_capita_income"] = pd.to_numeric(
    hardship["per_capita_income"], errors="coerce"
)

# STEP 5 — Hardship Index numeric
hardship["hardship_index"] = pd.to_numeric(
    hardship["hardship_index"], errors="coerce"
)

# Save cleaned file
hardship_out = os.path.join(CLEANED_DIR, "HardshipIndex_Cleaned.csv")
hardship.to_csv(hardship_out, index=False)
print(f"Saved cleaned Hardship Index to: {hardship_out} (rows: {len(hardship)})")

Loading Hardship Index from: /Users/nishantkhandhar/Desktop/Fall 25/DPA/Project/CSP571---Final-Project/Dataset/HardshipIndex.csv
Saved cleaned Hardship Index to: /Users/nishantkhandhar/Desktop/Fall 25/DPA/Project/CSP571---Final-Project/CleanedDataset/HardshipIndex_Cleaned.csv (rows: 77)


In [6]:
# ============================
# 4. AGGREGATE CRIME → COMMUNITY–YEAR
# ============================

# Basic community-year incident counts
community_year = (
    crime_all
    .groupby(["community_area", "year"])
    .size()
    .reset_index(name="total_incidents")
)

community_year_out = os.path.join(CLEANED_DIR, "Community_Year_Incidents.csv")
community_year.to_csv(community_year_out, index=False)
print(
    f"Saved community-year incident table to: {community_year_out} "
    f"(rows: {len(community_year)})"
)

Saved community-year incident table to: /Users/nishantkhandhar/Desktop/Fall 25/DPA/Project/CSP571---Final-Project/CleanedDataset/Community_Year_Incidents.csv (rows: 385)


In [7]:
# ============================
# 5. MERGE HARDSHIP ONTO COMMUNITY–YEAR
# ============================

final_df = community_year.merge(
    hardship[[
        "community_area",
        "unemployment_rate",
        "hardship_index",
        "per_capita_income",
    ]],
    on="community_area",
    how="left",
)

final_out = os.path.join(CLEANED_DIR, "Community_Year_With_Hardship.csv")
final_df.to_csv(final_out, index=False)
print(
    f"Saved final merged dataset to: {final_out} "
    f"(rows: {len(final_df)})"
)

print("\nDone. You are ready for EDA and modeling 🎯")


Saved final merged dataset to: /Users/nishantkhandhar/Desktop/Fall 25/DPA/Project/CSP571---Final-Project/CleanedDataset/Community_Year_With_Hardship.csv (rows: 385)

Done. You are ready for EDA and modeling 🎯
